In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from scipy.interpolate import Akima1DInterpolator
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

In [2]:
df_raw = pd.read_csv('../data/raw/dataset.csv')
df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], dayfirst=True)

df = pd.read_csv('../data/processed/dataset_long_pruned.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

dates       = sorted(df['datetime'].dt.date.unique())
EXPIRY_DATE = dates[-1]  # Jan 27

option_cols = [c for c in df_raw.columns if c not in ['datetime', 'underlying_price']]

print(f'Raw dataset      : {df_raw.shape}')
print(f'Long dataset     : {df.shape}')
print(f'Total missing    : {df["iv"].isna().sum():,}')
print(f'Expiry date      : {EXPIRY_DATE}')
print(f'Option columns   : {len(option_cols)}')

Raw dataset      : (975, 30)
Long dataset     : (27300, 17)
Total missing    : 5,460
Expiry date      : 2026-01-27
Option columns   : 28


In [3]:
FEATURE_COLS = [
    'iv_roll_mean_10',
    'iv_roll_mean_5',
    'iv_neighbor_mean',
    'wide_iv_neighbor_mean',
    'strike',
    'iv_neighbor_+1',
    'iv_neighbor_-1',
    'iv_neighbor_+2',
    'iv_neighbor_-2',
    'dist_from_atm_pct',
    'moneyness',
    'log_moneyness'
]

class IVSurfaceMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.1):
        super().__init__()
        layers = []; prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp(X_tr, y_tr, X_ev, y_ev, epochs=500, batch_size=256, patience=40):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr); X_ev_s = scaler.transform(X_ev)
    Xtt = torch.tensor(X_tr_s, dtype=torch.float32)
    ytt = torch.tensor(y_tr,   dtype=torch.float32)
    Xvt = torch.tensor(X_ev_s, dtype=torch.float32)
    yvt = torch.tensor(y_ev,   dtype=torch.float32)
    loader  = DataLoader(TensorDataset(Xtt, ytt), batch_size=batch_size, shuffle=True)
    model   = IVSurfaceMLP(X_tr.shape[1])
    opt     = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.MSELoss()
    best_loss, best_state, no_imp = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad(): vl = loss_fn(model(Xvt), yvt).item()
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= patience: break
    model.load_state_dict(best_state)
    return model, scaler

def predict_mlp(model, scaler, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(scaler.transform(X), dtype=torch.float32)).numpy()

In [4]:
# Train on ALL observed rows — no train/val split, use full data
# We use last 15% as internal eval for early stopping only
obs_all = df[df['iv'].notna()]
X_all   = obs_all[FEATURE_COLS].fillna(0).values
y_all   = obs_all['iv'].values
cut     = int(0.85 * len(X_all))

print(f'Training MLP on {len(X_all):,} observed samples...')
mlp_model, mlp_scaler = train_mlp(
    X_all[:cut], y_all[:cut],
    X_all[cut:], y_all[cut:]
)

Training MLP on 21,840 observed samples...


In [5]:
def predict_interp(mask_rows, df_context):

    preds = []

    for _, row in mask_rows.iterrows():

        ts_data = df_context[
            (df_context['datetime'] == row['datetime']) &
            (df_context['option_type'] == row['option_type']) &
            (df_context['iv'].notna())
        ].sort_values('strike')

        if len(ts_data) >= 5:

            try:

                f = Akima1DInterpolator(
                    ts_data['strike'].values,
                    ts_data['iv'].values
                )

                pred = f(row['strike'])

                if np.isnan(pred):
                    preds.append(np.nan)
                else:
                    preds.append(float(pred))

            except:
                preds.append(np.nan)

        else:
            preds.append(np.nan)

    return np.array(preds)


def predict_cascaded(missing_rows, df_context, mlp_model, mlp_scaler, expiry_date):
    preds       = np.full(len(missing_rows), np.nan)
    expiry_mask = missing_rows['datetime'].dt.date == expiry_date

    # Expiry day → interpolation first
    if expiry_mask.any():
        p_interp = predict_interp(missing_rows[expiry_mask], df_context)
        preds[expiry_mask.values] = p_interp

    # Non-expiry days → MLP
    non_expiry_mask = ~expiry_mask
    if non_expiry_mask.any():
        X = missing_rows[non_expiry_mask][FEATURE_COLS].fillna(0).values
        preds[non_expiry_mask.values] = predict_mlp(mlp_model, mlp_scaler, X)

    # Fallback: any NaN remaining (interp failed) → MLP
    still_nan = np.isnan(preds)
    if still_nan.any():
        X_fb = missing_rows[still_nan][FEATURE_COLS].fillna(0).values
        preds[still_nan] = predict_mlp(mlp_model, mlp_scaler, X_fb)
        print(f'  Fallback applied to {still_nan.sum()} cells')

    return preds


# Predict all missing cells
missing_rows = df[df['iv'].isna()].copy()
print(f'Predicting {len(missing_rows):,} missing cells...')

predictions = predict_cascaded(
    missing_rows = missing_rows,
    df_context   = df,
    mlp_model    = mlp_model,
    mlp_scaler   = mlp_scaler,
    expiry_date  = EXPIRY_DATE
)

print(f'  NaN in predictions : {np.isnan(predictions).sum()}')
print(f'  Min prediction     : {np.nanmin(predictions):.4f}')
print(f'  Max prediction     : {np.nanmax(predictions):.4f}')
print(f'  Mean prediction    : {np.nanmean(predictions):.4f}')

Predicting 5,460 missing cells...
  Fallback applied to 73 cells
  NaN in predictions : 0
  Min prediction     : 0.0850
  Max prediction     : 5.0607
  Mean prediction    : 0.1837


## Reconstructing filled_dataset.csv

In [6]:
# Fill predictions back into the raw wide-format dataframe
df_filled = df_raw.copy()

# Map predictions back using (datetime, contract) as key
missing_rows_copy = missing_rows.copy()
missing_rows_copy['predicted_iv'] = predictions

filled_count = 0
for _, row in missing_rows_copy.iterrows():
    dt       = row['datetime']
    contract = row['contract']  # e.g. 'NIFTY27JAN2625200CE'
    pred_val = row['predicted_iv']

    # Find matching row in wide-format df
    idx = df_filled[df_filled['datetime'] == dt].index
    if len(idx) > 0 and contract in df_filled.columns:
        df_filled.loc[idx[0], contract] = pred_val
        filled_count += 1

print(f'Cells filled in wide-format df : {filled_count:,}')
print(f'Remaining NaN in df_filled     : {df_filled[option_cols].isna().sum().sum()}')

Cells filled in wide-format df : 5,460
Remaining NaN in df_filled     : 0


In [ ]:
remaining_nan = df_filled[option_cols].isna().sum().sum()
assert remaining_nan == 0, f'Still {remaining_nan} NaN value'

df_filled[option_cols] = df_filled[option_cols].clip(lower=0.0001)

df_filled.to_csv('../outputs/2_submissions/filled_dataset.csv', index=False)

print(f'filled_dataset.csv saved')
print(f'Shape  : {df_filled.shape}')
print(f'NaN    : {df_filled[option_cols].isna().sum().sum()}')
print(f'Min IV : {df_filled[option_cols].min().min():.4f}')
print(f'Max IV : {df_filled[option_cols].max().max():.4f}')

filled_dataset.csv saved
Shape  : (975, 30)
NaN    : 0
Min IV : 0.0168
Max IV : 5.3848


In [9]:
SEPARATOR = '||'

def generate_solution(filled_path, output_path, original_path):
    original = pd.read_csv(original_path)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != 'datetime']
    rows = []

    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, 'datetime']
            uid = f'{dt}{SEPARATOR}{col}'
            val = filled.loc[idx, col]
            rows.append({'id': uid, 'value': val})

    solution = pd.DataFrame(rows, columns=['id', 'value'])
    solution = solution.sort_values('id').reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f'Solution saved → {output_path}  ({len(solution)} rows)')
    return solution

solution = generate_solution(
    filled_path   = '../outputs/2_submissions/filled_dataset.csv',
    output_path   = '../outputs/2_submissions/submission.csv',
    original_path = '../data/raw/dataset.csv'
)
solution.head(10)

Solution saved → ../outputs/2_submissions/submission.csv  (5460 rows)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.176583
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.112437
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.102676
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.166745
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.173178
5,07-01-2026 09:20||NIFTY27JAN2624800PE,0.123859
6,07-01-2026 09:20||NIFTY27JAN2625000PE,0.114695
7,07-01-2026 09:20||NIFTY27JAN2625300CE,0.110309
8,07-01-2026 09:20||NIFTY27JAN2625400CE,0.116643
9,07-01-2026 09:20||NIFTY27JAN2625800CE,0.104644


Final Verification

In [13]:
sub = pd.read_csv('../outputs/2_submissions/submission.csv')

# Check 1: Row count
expected_rows = 5460
status = 'PASS' if len(sub) == expected_rows else 'FAIL'
print(f'[{status}] Row count: {len(sub)} (expected {expected_rows})')

# Check 2: Columns
status = 'PASS' if list(sub.columns) == ['id', 'value'] else 'FAIL'
print(f'[{status}] Columns: {list(sub.columns)}')

# Check 3: No NaN values
status = 'PASS' if sub['value'].isna().sum() == 0 else 'FAIL'
print(f'[{status}] NaN values: {sub["value"].isna().sum()}')

# Check 4: No negative or zero IV
status = 'PASS' if (sub['value'] > 0).all() else 'FAIL'
print(f'[{status}] All values positive: {(sub["value"] > 0).all()}')

# Check 5: ID format check (sample)
sample_id = sub['id'].iloc[0]
status = 'PASS' if '||' in sample_id else 'FAIL'
print(f'[{status}] ID format sample: {sample_id}')

# Check 6: Value range sanity
print(f'[INFO] Value range: {sub["value"].min():.4f} – {sub["value"].max():.4f}')
print(f'[INFO] Value mean : {sub["value"].mean():.4f}')

print()
all_pass = len(sub)==expected_rows and list(sub.columns)==['id','value'] and sub['value'].isna().sum()==0 and (sub['value']>0).all()
if all_pass:
    print('All checks passed')
else:
    print('Not all checks are passed')

[PASS] Row count: 5460 (expected 5460)
[PASS] Columns: ['id', 'value']
[PASS] NaN values: 0
[PASS] All values positive: True
[PASS] ID format sample: 07-01-2026 09:15||NIFTY27JAN2624100PE
[INFO] Value range: 0.0850 – 5.0607
[INFO] Value mean : 0.1837

All checks passed
